In [1]:
import pandas as pd
import numpy as np
import pickle
import os

print("Libraries loaded")

Libraries loaded


In [2]:
X_train = pd.read_csv("../data-pipeline/data/processed/X_train.csv")
X_test  = pd.read_csv("../data-pipeline/data/processed/X_test.csv")
y_train = pd.read_csv("../data-pipeline/data/processed/y_train.csv")["class"]
y_test  = pd.read_csv("../data-pipeline/data/processed/y_test.csv")["class"]

with open("../data-pipeline/data/processed/minimal_feature_set.pkl", "rb") as f:
    features = pickle.load(f)

X_train_efs = X_train[features]
X_test_efs  = X_test[features]

print(f"Features: {features}")
print(f"Training: {X_train_efs.shape}")
print(f"Test:     {X_test_efs.shape}")
print(f"Classes:  {sorted(y_train.unique().tolist())}")

Features: ['Header_Length', 'Number', 'ack_flag_number', 'TCP', 'ack_count', 'Tot size', 'AVG']
Training: (1916259, 7)
Test:     (354705, 7)
Classes:  ['Benign', 'DDoS', 'Reconnaissance']


In [3]:
classes = ['Benign', 'DDoS', 'Reconnaissance']

# Per-class means for each feature
means = {}
for cls in classes:
    mask = y_train == cls
    means[cls] = X_train_efs[mask].mean()

means_df = pd.DataFrame(means).T
print("Per-class feature means (Min-Max scaled 0-1):")
print(means_df.round(4).to_string())

# Derive thresholds
print("Derived thresholds:")
print(f"{'Feature':<20} {'DDoS threshold':<18} {'Recon threshold':<18} {'DDoS direction':<16} {'Recon direction'}")
print("-" * 85)

thresholds = {}
for feat in features:
    b  = means['Benign'][feat]
    d  = means['DDoS'][feat]
    r  = means['Reconnaissance'][feat]

    # DDoS threshold: midpoint between Benign and DDoS means
    ddos_thresh = (b + d) / 2
    ddos_op = '>' if d > b else '<'

    # Recon threshold: midpoint between Benign and Recon means
    recon_thresh = (b + r) / 2
    recon_op = '>' if r > b else '<'

    thresholds[feat] = {
        'ddos': ddos_thresh, 'ddos_op': ddos_op,
        'recon': recon_thresh, 'recon_op': recon_op
    }
    print(f"{feat:<20} {ddos_thresh:<18.4f} {recon_thresh:<18.4f} {ddos_op:<16} {recon_op}")

Per-class feature means (Min-Max scaled 0-1):
                Header_Length  Number  ack_flag_number     TCP  ack_count  Tot size     AVG
Benign                 0.4554  0.0816           0.7978  0.8092     0.0798    0.0582  0.0582
DDoS                   0.0459  0.9996           0.0025  0.0028     0.0025    0.0001  0.0001
Reconnaissance         0.3842  0.0816           0.5762  0.7964     0.0576    0.0171  0.0171
Derived thresholds:
Feature              DDoS threshold     Recon threshold    DDoS direction   Recon direction
-------------------------------------------------------------------------------------
Header_Length        0.2507             0.4198             <                <
Number               0.5406             0.0816             >                >
ack_flag_number      0.4001             0.6870             <                <
TCP                  0.4060             0.8028             <                <
ack_count            0.0411             0.0687             <                

In [4]:
# Build rules from computed thresholds
# Order matters: DDoS rules first, then Recon, then catch-all Benign
rules = []

# --- DDoS rules ---
for feat in ['Number', 'ack_count', 'ack_flag_number', 'Tot size']:
    t = thresholds[feat]
    rules.append({
        'feature': feat,
        'op': t['ddos_op'],
        'threshold': round(t['ddos'], 4),
        'class': 'DDoS',
        'class_id': 1,
        'explanation': f"DDoS detected: {feat} is {'above' if t['ddos_op']=='>' else 'below'} normal range, consistent with volumetric flooding."
    })

# --- Reconnaissance rules ---
for feat in ['TCP', 'Header_Length', 'AVG']:
    t = thresholds[feat]
    rules.append({
        'feature': feat,
        'op': t['recon_op'],
        'threshold': round(t['recon'], 4),
        'class': 'Reconnaissance',
        'class_id': 2,
        'explanation': f"Probe detected: {feat} is {'above' if t['recon_op']=='>' else 'below'} normal range, consistent with network reconnaissance."
    })

# --- Catch-all Benign rule (always fires last) ---
rules.append({
    'feature': None,
    'op': None,
    'threshold': None,
    'class': 'Benign',
    'class_id': 0,
    'explanation': "No threat detected: traffic features are within normal operating ranges."
})

rules_df = pd.DataFrame(rules)
print(f"Total rules: {len(rules)}")
print("Rule table:")
for i, r in enumerate(rules):
    if r['feature']:
        print(f"  Rule {i+1:2d}: IF {r['feature']:<20} {r['op']} {r['threshold']:.4f} → {r['class']}")
    else:
        print(f"  Rule {i+1:2d}: DEFAULT → {r['class']}")

Total rules: 8
Rule table:
  Rule  1: IF Number               > 0.5406 → DDoS
  Rule  2: IF ack_count            < 0.0411 → DDoS
  Rule  3: IF ack_flag_number      < 0.4001 → DDoS
  Rule  4: IF Tot size             < 0.0291 → DDoS
  Rule  5: IF TCP                  < 0.8028 → Reconnaissance
  Rule  6: IF Header_Length        < 0.4198 → Reconnaissance
  Rule  7: IF AVG                  < 0.0376 → Reconnaissance
  Rule  8: DEFAULT → Benign


In [5]:
# Class-conditional coverage: rules explain the MODEL's prediction, not override it

def get_explanation(predicted_class_name, feature_vector, rule_list):
    """Find the best matching rule for the given predicted class."""
    # First try to find a feature-based rule for this class
    for rule in rule_list:
        if rule['class'] != predicted_class_name:
            continue
        if rule['feature'] is None:  # class-specific catch-all
            return rule
        val = feature_vector[rule['feature']]
        if rule['op'] == '>' and val > rule['threshold']:
            return rule
        if rule['op'] == '<' and val < rule['threshold']:
            return rule
    # Absolute fallback: return the class catch-all regardless
    for rule in rule_list:
        if rule['class'] == predicted_class_name and rule['feature'] is None:
            return rule
    return rule_list[-1]  # default Benign

# Load the TFLite model to get model predictions
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="../model/outputs/model.tflite")
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
scale      = input_details[0]['quantization'][0]
zero_point = input_details[0]['quantization'][1]

CLASS_NAMES = ['Benign', 'DDoS', 'Reconnaissance']

covered = 0
sample_outputs = []

print("Running class-conditional coverage test (this may take a few minutes)...")

for i in range(len(X_test_efs)):
    row = X_test_efs.iloc[i]

    # Step 1: model predicts the class
    sample = row.values.reshape(1, -1).astype(np.float32)
    sample_int8 = (sample / scale + zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    pred_class_id   = int(np.argmax(output))
    pred_class_name = CLASS_NAMES[pred_class_id]

    # Step 2: rules explain WHY the model made that prediction
    matched_rule = get_explanation(pred_class_name, row, rules)

    if matched_rule:
        covered += 1
        if i < 5:  # show first 5 examples
            sample_outputs.append(
                f"  Sample {i+1}: Model={pred_class_name:15s} | "
                f"Rule fired on '{matched_rule['feature'] or 'catch-all'}' | "
                f"Explanation: {matched_rule['explanation'][:60]}..."
            )

coverage_pct = (covered / len(X_test_efs)) * 100

# Compute class agreement properly
# In this architecture it should always be 100% by design, but we verify it
class_match = sum(
    1 for i in range(len(X_test_efs))
    for row, true_pred in [(X_test_efs.iloc[i], None)]
    if True  # placeholder — we track this below
)

# Re-run agreement check properly
agreement_count = 0
for i in range(len(X_test_efs)):
    row = X_test_efs.iloc[i]
    sample = row.values.reshape(1, -1).astype(np.float32)
    sample_int8 = (sample / scale + zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    pred_class_name = CLASS_NAMES[int(np.argmax(output))]
    matched_rule = get_explanation(pred_class_name, X_test_efs.iloc[i], rules)
    if matched_rule['class'] == pred_class_name:
        agreement_count += 1

agreement_pct = (agreement_count / len(X_test_efs)) * 100

print(f"Test samples:    {len(X_test_efs):,}")
print(f"Coverage:        {coverage_pct:.2f}%")
print(f"Class agreement: {agreement_pct:.2f}%")


Running class-conditional coverage test (this may take a few minutes)...
Test samples:    354,705
Coverage:        100.00%
Class agreement: 99.75%


In [6]:
# Generate xai_rules.h

os.makedirs("outputs", exist_ok=True)

lines = []
lines.append("// Auto-generated XAI rule table")
lines.append("// Source: 05_xai_rules.ipynb")
lines.append("// Features: " + str(features))
lines.append("")
lines.append("#ifndef XAI_RULES_H")
lines.append("#define XAI_RULES_H")
lines.append("")
lines.append("struct XAIRule {")
lines.append("  const char* feature;")
lines.append("  float threshold;")
lines.append("  bool greater_than;")
lines.append("  int predicted_class;")
lines.append("  const char* explanation;")
lines.append("};")
lines.append("")
lines.append(f"const int NUM_RULES = {len(rules)};")
lines.append("const XAIRule xai_rules[] = {")

rule_lines = []
for r in rules:
    if r['feature'] is None:
        rule_lines.append(f'  {{NULL, 0.0f, true, {r["class_id"]}, "{r["explanation"]}\"}}')
    else:
        gt = 'true' if r['op'] == '>' else 'false'
        rule_lines.append(f'  {{"{r["feature"]}", {r["threshold"]}f, {gt}, {r["class_id"]}, "{r["explanation"]}\"}}')

lines.append(",\n".join(rule_lines))
lines.append("};")
lines.append("")
lines.append("#endif // XAI_RULES_H")

full = "\n".join(lines)

with open("outputs/xai_rules.h", "w") as f:
    f.write(full)

print("xai_rules.h generated successfully")
print(f"Total rules: {len(rules)}")
print("\nFile preview:")
print(full[:800])

xai_rules.h generated successfully
Total rules: 8

File preview:
// Auto-generated XAI rule table
// Source: 05_xai_rules.ipynb
// Features: ['Header_Length', 'Number', 'ack_flag_number', 'TCP', 'ack_count', 'Tot size', 'AVG']

#ifndef XAI_RULES_H
#define XAI_RULES_H

struct XAIRule {
  const char* feature;
  float threshold;
  bool greater_than;
  int predicted_class;
  const char* explanation;
};

const int NUM_RULES = 8;
const XAIRule xai_rules[] = {
  {"Number", 0.5406f, true, 1, "DDoS detected: Number is above normal range, consistent with volumetric flooding."},
  {"ack_count", 0.0411f, false, 1, "DDoS detected: ack_count is below normal range, consistent with volumetric flooding."},
  {"ack_flag_number", 0.4001f, false, 1, "DDoS detected: ack_flag_number is below normal range, consistent with volumetric flooding."},
  {"Tot size", 0.0291f, false,


In [7]:
import shutil

os.makedirs("../firmware/src", exist_ok=True)

# Copy both header files into firmware/src/
shutil.copy("outputs/xai_rules.h", "../firmware/src/xai_rules.h")
shutil.copy("outputs/model.h",     "../firmware/src/model.h")

print("Copied to firmware/src/:")
print("  model.h")
print("  xai_rules.h")
print("Firmware folder contents:")
for f in sorted(os.listdir("../firmware/src")):
    print(f"  {f}")

Copied to firmware/src/:
  model.h
  xai_rules.h
Firmware folder contents:
  main.cpp
  model.h
  xai_rules.h


In [8]:
import pandas as pd
import pickle
import numpy as np

X_test = pd.read_csv("../data-pipeline/data/processed/X_test.csv")
y_test = pd.read_csv("../data-pipeline/data/processed/y_test.csv")["class"]

with open("../data-pipeline/data/processed/minimal_feature_set.pkl", "rb") as f:
    features = pickle.load(f)

X_test_efs = X_test[features]

print("// Paste these into main.cpp as TEST_VECTORS")
print("const float TEST_VECTORS[3][7] = {")
for i, cls in enumerate(['Benign', 'DDoS', 'Reconnaissance']):
    idx = y_test[y_test == cls].index[0]
    vals = X_test_efs.loc[idx].values
    vals_str = ", ".join([f"{v:.4f}f" for v in vals])
    comma = "," if i < 2 else ""
    print(f"  {{{vals_str}}}{comma}  // {cls}")
print("}};")
print()
print("// Feature order:", features)


// Paste these into main.cpp as TEST_VECTORS
const float TEST_VECTORS[3][7] = {
  {0.5133f, 0.0816f, 1.0000f, 1.0000f, 0.1000f, 0.1403f, 0.1403f},  // Benign
  {0.1333f, 1.0000f, 0.0000f, 0.0000f, 0.0000f, 0.0000f, 0.0000f},  // DDoS
  {0.4533f, 0.0816f, 0.8000f, 0.8000f, 0.0800f, 0.0026f, 0.0026f}  // Reconnaissance
}};

// Feature order: ['Header_Length', 'Number', 'ack_flag_number', 'TCP', 'ack_count', 'Tot size', 'AVG']


In [1]:
import shutil, os
os.makedirs('../firmware/src', exist_ok=True)
shutil.copy('outputs/xai_rules.h', '../firmware/src/xai_rules.h')
shutil.copy('outputs/model.h',     '../firmware/src/model.h')
print('Copied updated model.h and xai_rules.h to firmware/src/')

Copied updated model.h and xai_rules.h to firmware/src/
